In [42]:
import gc
import os
import pickle
import random
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score as APS
from sklearn.model_selection import ShuffleSplit, StratifiedShuffleSplit
import duckdb
import pandas as pd
import mapply
from sklearn.metrics import average_precision_score, precision_recall_curve
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast
from transformers import AutoConfig, AutoTokenizer, AutoModel, DataCollatorWithPadding
import torch
import torch.nn as nn
import torch.nn.functional as F
import importlib
from torch.optim import Adam, AdamW
import matplotlib.pyplot as plt
import gc
import matplotlib as mpl
import json
from sklearn.metrics import classification_report, cohen_kappa_score
from sklearn.utils import class_weight
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
from sklearn.preprocessing import StandardScaler
# from keras.layers import Layer
from rdkit import Chem
# from tensorflow.keras.layers import LeakyReLU
class Wrapper:
    def __init__(self, method_name, module_name):
        self.method_name = method_name
        self.module = importlib.import_module(module_name)

    @property
    def method(self):
        return getattr(self.module, self.method_name)

    def __call__(self, *args, **kwargs):
        return self.method(*args, **kwargs)
        
wrapped_mol_from_smiles = Wrapper("MolFromSmiles", "rdkit.Chem")

mapply.init(
    n_workers=10,#was 5 before, trying 10,
    progressbar=True,
)

wrapped_has_substract_match = Wrapper("HasSubstructMatch", "rdkit.Chem")

# calc some descriptors
def calc_descriptors(smile):
    mol = wrapped_mol_from_smiles(smile)
    descr=[round(Descriptors.MolLogP(mol),4),
                  round(Descriptors.TPSA(mol),4),
                  Descriptors.NHOHCount(mol),
                  Descriptors.NOCount(mol),
                  Descriptors.NumHAcceptors(mol),
                  Descriptors.NumHDonors(mol),
                  Descriptors.NumRotatableBonds(mol),
                  Descriptors.NumHeteroatoms(mol),
                  round(Descriptors.FractionCSP3(mol),4),
                  round(Descriptors.ExactMolWt(mol),4),
                  rdMolDescriptors.CalcNumAromaticRings(mol)]
    return np.array(descr,dtype=np.float16)

def get_unique_values(list_of_lists):
    unique_values = []
    for inner_list in list_of_lists:
        for value in inner_list:
            unique_values.append(value)
    return unique_values

In [2]:
all_data = pd.read_csv('all_data_filtered.csv',index_col=[0])

In [3]:
all_data

,Id,smiles,sol_category,sizes
0,EOS12286,Cc1nc(N2CCN(C(=O)Nc3ccc(F)cc3F)CC2)cc(-n2ccnc2)n1,0,49
1,EOS85869,CCN(CC)[C@H]1CCN(C(=O)Cc2nc(C(C)C)c(C)s2)C1,0,43
2,EOS85435,CNC(=O)CNC(=O)c1c(-n2cccc2)sc(C)c1C,0,35
3,EOS102302,CC(C)(C)c1ccc(CSc2cnn(C(C)(C)C)c(=O)c2Cl)cc1,0,44
4,EOS64213,CC[C@H](NC(=O)c1ccnc(-n2ccnc2)c1)c1ccccc1OC,0,43
...,...,...,...,...
70559,EOS37839,O=C(NCCCc1nc(=O)[nH][nH]1)[C@H]1CCC(F)(F)C1,2,43
70560,EOS2088,Cc1ccc(C(=O)NC2CCCC2)cc1S(=O)(=O)N1CCOCC1,2,41
70561,EOS10587,COCCN1CCC(CN(C)S(=O)(=O)c2cccc(C(F)(F)F)c2)C1,2,45
70562,EOS40533,O=C(Nc1ccc(F)cc1)NC1CCN(C(=O)Cc2cnn(-c3ccccc3)...,2,52


In [4]:
print(all_data.sizes.max())
print(all_data.sizes.min())
print(all_data.sizes.mean())

95
18
42.714812085482684


In [36]:
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast
from tokenizers.models import WordLevel,BPE

from tokenizers.pre_tokenizers import Whitespace,Split,ByteLevel

from tokenizers.normalizers import Lowercase, NFKC

enc = { '[PAD]':0,
        'Br':1, 'C':2, 'N':3, 'O':4, 'H':5, 'S':6, 'F':7, 'Cl':8, 'B':9, 'I':10, 
        's':11,'o':12, 'c':13, 'n':14, 'i':15, # is Atomic
        '.':16 ,'=':17 ,'#':18, # bond
        '/':19, # direction
        '-':20, '+': 21, # charge
        '[':22,']':23, # Atomic mass
        '(':24,')':25, # Branches
        '@@':26, '@':27, # tetrahedron
        '1':28,'2':29,'3':30,'4':31,'5':32,'6':33,'7':34,'8':35,'9':36,'Na':37,'P':38,'[UNK]':39
      }



tokenizer = Tokenizer(BPE(vocab=enc, unk_token="[UNK]",merges=[('@','@')]))

tokenizer_fast = PreTrainedTokenizerFast(tokenizer_object=tokenizer)
tokenizer_fast.add_special_tokens({'pad_token': '[PAD]'})
tokenizer_fast.add_special_tokens({'additional_special_tokens':['Br','Cl','Na']})

len(set(enc.values()))

40

In [38]:
def normalize_smile(smile):
    mol = wrapped_mol_from_smiles(smile)
    smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
    return smiles

def encode_smile(smile):
    enc = tokenizer_fast.batch_encode_plus(
        [smile],
        max_length=95,
        padding='max_length',
        return_tensors='np',
        truncation=True
    )['input_ids']
    return enc.astype(np.uint8)

In [7]:


# all_data['smiles'] = all_data['smiles'].apply(normalize_smile)
# all_data['enc_smiles'] = all_data['smiles'].apply(encode_smile)
descr = []
# calc some descriptors
for m in tqdm(all_data.smiles):
    descr.append([Descriptors.MolLogP(Chem.MolFromSmiles(m)),
                  Descriptors.TPSA(Chem.MolFromSmiles(m)),
                  Descriptors.NHOHCount(Chem.MolFromSmiles(m)),
                  Descriptors.NOCount(Chem.MolFromSmiles(m)),
                  Descriptors.NumHAcceptors(Chem.MolFromSmiles(m)),
                  Descriptors.NumHDonors(Chem.MolFromSmiles(m)),
                  Descriptors.NumRotatableBonds(Chem.MolFromSmiles(m)),
                  Descriptors.NumHeteroatoms(Chem.MolFromSmiles(m)),
                  Descriptors.FractionCSP3(Chem.MolFromSmiles(m)),
                  Descriptors.ExactMolWt(Chem.MolFromSmiles(m)),
                  rdMolDescriptors.CalcNumAromaticRings(Chem.MolFromSmiles(m))])

descr = np.asarray(descr)

100%|███████████████████████████████████████████████████████████████████████████| 70564/70564 [02:49<00:00, 416.46it/s]


In [8]:
descr.shape

(70564, 11)

In [9]:
# scaler_descr = StandardScaler()

In [10]:
scaler_descr = torch.load('rdkit_descriptor_scaler_sol')
descr_scaled = scaler_descr.transform(descr)

In [11]:
# torch.save(scaler_descr,'rdkit_descriptor_scaler_sol')

In [11]:
all_data['norm_smiles'] = all_data['smiles'].apply(normalize_smile)
all_data['enc_smiles'] = all_data['norm_smiles'].apply(encode_smile)

In [12]:
unk_counter = 0
for item in range(len(all_data['enc_smiles'].values)):
    if 39 in all_data['enc_smiles'].values[item]:
        unk_counter+=1
print(unk_counter)

5


In [13]:
# V6bs
import torch.nn.functional as F
from torch.nn import Linear, Conv1d, Embedding, AdaptiveMaxPool1d, AdaptiveAvgPool1d
from torch_geometric.nn import BatchNorm


class CNN1D(torch.nn.Module):
    def __init__(self,
                 conv_filters = 64,
                 conv_layers = 5,
                 emb_dim = 256,
                 kernel_size = 5,
                 linear_neurons=1024):

        super(CNN1D, self).__init__()
        
        self.emb_dim = emb_dim
        
        self.conv_filters = conv_filters

        self.num_conv1d_layers = conv_layers
        
        self.smile_emb = Embedding(num_embeddings=40, embedding_dim=emb_dim)

        self.conv1d_layers = nn.ModuleList()

        for i in range(conv_layers):
            if i == 0:
                self.conv1d_layers.append(
                    Conv1d(in_channels=emb_dim, out_channels=conv_filters * (i+1), kernel_size=kernel_size, stride=1))
            else:
                self.conv1d_layers.append(
                Conv1d(in_channels=conv_filters * i, out_channels=conv_filters * (i+1), kernel_size=kernel_size, stride=1))

        self.maxpool = AdaptiveMaxPool1d(1)
        self.avgpool = AdaptiveAvgPool1d(1)


        self.conv_act = nn.ReLU()
        self.batchnorm = nn.BatchNorm1d(11 + (conv_filters * (conv_layers) * 2))

        # self.actfc = nn.LeakyReLU(negative_slope=0.1)

        # self.gated_mlp = Gated_MLP_block(384)

        self.out_mlp = nn.Sequential(Linear(in_features= 11 + (conv_filters * (conv_layers) * 2), out_features=linear_neurons),
                                     nn.BatchNorm1d(linear_neurons),
                                     nn.LeakyReLU(),
                                     nn.Dropout(0.2),
                                    Linear(in_features=linear_neurons, out_features=linear_neurons //2),
                                     nn.BatchNorm1d(linear_neurons // 2),
                                     nn.LeakyReLU(),
                                     nn.Dropout(0.2),
                                     Linear(in_features=linear_neurons // 2, out_features=linear_neurons //4),
                                     nn.BatchNorm1d(linear_neurons // 4),
                                     nn.LeakyReLU(),
                                     nn.Dropout(0.2),
                                     Linear(in_features=linear_neurons // 4, out_features=linear_neurons // 8),
                                     nn.BatchNorm1d(linear_neurons // 8),
                                     nn.LeakyReLU(),
                                     nn.Dropout(0.2),
                                     )

        self.layer_out = Linear(in_features=linear_neurons // 8, out_features=3)

    def forward(self, smile_vec,rdkit):

        smile_emb = self.smile_emb(smile_vec)

        conv_out = smile_emb.reshape(len(smile_emb), self.emb_dim, -1)

        for i in range(self.num_conv1d_layers):
            conv_out = self.conv1d_layers[i](conv_out)
            conv_out = self.conv_act(conv_out)

        conv_out_max = self.maxpool(conv_out).reshape(len(conv_out), self.conv_filters * self.num_conv1d_layers)
        conv_out_avg = self.avgpool(conv_out).reshape(len(conv_out), self.conv_filters * self.num_conv1d_layers)

        out = torch.cat((conv_out_max,conv_out_avg,rdkit), dim=1)
        out = self.batchnorm(out)

        out = self.out_mlp(out)

        out = self.layer_out(out)

        return out

In [14]:
def get_loader(smile_emb,rdkit_f,y=None, with_y=True, shuffle=False, batch_size=4096):
    if with_y:
        train = torch.utils.data.TensorDataset(torch.from_numpy(smile_emb).long(),
                                               torch.from_numpy(rdkit_f).float(),
                                           torch.from_numpy(y.flatten()).long())
    if not with_y:
        train = torch.utils.data.TensorDataset(torch.from_numpy(smile_emb).long(),
                                              torch.from_numpy(rdkit_f).float())
        
    train_loader = torch.utils.data.DataLoader(train, 
                                               batch_size=batch_size,shuffle=shuffle)
    return train_loader

In [14]:
def train_epoch(model, optimizer, train_loader, criterion, device):
    model.train()
    for smile_emb,rdkit_f,y in train_loader:
  
        out = model(smile_emb.to(device),rdkit_f.to(device))
        loss = criterion(out, y.to(device))

        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def evaluate_loss(loader, model, criterion, device, metric=None):
    model.eval()
    with torch.no_grad():
        loss = 0
        j = 0
        sg = nn.Softmax()
        for smile_emb,rdkit_f,y in loader:
            out = model(smile_emb.to(device),rdkit_f.to(device))
            loss += criterion(out, y.to(device))
            if metric is not None:
                metric.update(sg(out), y.to(device).long())
            j += 1
        return (loss / j)


def train_m_v3(model, opt, train_loader, val_loader, criterion, n_epochs, device, name, 
               metric=None, early_stop=5,scheduler=None, verbose=True):
    train_log = []
    val_log = []
    val_metric_log = []
    train_metric_log = []
    val_min = np.inf
    train_min = np.inf
    epoch_min = 0
    fixed_epoch_val = 0
    fixed_epoch_train = 0
    val_allowed_maximum = np.inf
    early_counter = 0

    for epoch in tqdm(range(n_epochs)):
        train_epoch(model, opt, train_loader, criterion, device)
        train_loss = evaluate_loss(train_loader, model, criterion, device, metric)
        
        if metric is not None:
            train_metric = metric.compute().cpu().numpy().item()
            metric.reset()
        elif metric is None:
            train_metric = torch.tensor([0])

        val_loss = evaluate_loss(val_loader, model,
                                 criterion, device, metric)
        if metric is not None:
            val_metric = metric.compute().cpu().numpy().item()
            metric.reset()

        elif metric is None:
            train_metric = torch.tensor([0])
        
        if scheduler is not None:
            scheduler.step(val_loss)
            
        train_log.append(train_loss.cpu().numpy().item())
        val_log.append(val_loss.cpu().numpy().item())

        train_metric_log.append(train_metric)
        val_metric_log.append(val_metric)
        
        if val_loss < val_min:
            val_min = val_loss.cpu().numpy().item()
            epoch_val_absolute_min = epoch
            val_allowed_maximum = val_loss * 1.0001 #removed allowance for validation fluctuation

        if train_loss < train_min:
            train_min = train_loss

        if val_loss <= val_allowed_maximum:
            epoch_min = epoch
            fixed_epoch_val = val_loss.cpu().numpy().item()
            fixed_epoch_train = train_min.cpu().numpy().item()
            torch.save(model.state_dict(), name)
            torch.save({
                'epoch': epoch_min,
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss_history': train_log,
                'val_loss_history': val_log,
                'train_metric_history': train_metric_log,
                'val_metric_history': val_metric_log,
            }, name + 'training_optimizer_logs_')
            early_counter = 0

        if val_loss > val_allowed_maximum:
            early_counter += 1

        if verbose:
            # if epoch % 1 == 0:
            print(f'Epoch [{epoch + 1}/{n_epochs}], Loss (train/test): {train_loss}/{val_loss},'  
            f'AP (train/test): {train_metric_log[-1]}/{val_metric_log[-1]}')

        res_dict = {'min_loss_epoch': epoch_min,
                    'train_min': train_min.cpu().numpy().item(),
                    'train_loss_history': train_log,
                    'val_loss_history': val_log,
                    'val_min': val_min,
                    'final_val': fixed_epoch_val,
                    'final_train': fixed_epoch_train,
                'train_metric_history': train_metric_log,
                'val_metric_history': val_metric_log}

        if early_counter == early_stop:
            print(
                f'Early stopping after epoch {epoch}, model saved at epoch {epoch_min}, validation loss {fixed_epoch_val}.')
            return res_dict

    return res_dict

In [15]:
indep_idx = []
with open(f'solubility_indep_ids.txt', 'r') as fp:
    for line in fp.read().splitlines():
        indep_idx.append(line)
ind_id = all_data[all_data.Id.isin(indep_idx)].index

In [26]:
# train_idx = []
# with open(f'solubility_train_id_{0}.txt', 'r') as fp:
#     for line in fp.read().splitlines():
#         train_idx.append(line)

In [28]:
# tmp = all_data[all_data.Id.isin(indep_idx)]

In [29]:
# tmp

,Id,smiles,sol_category,sizes,norm_smiles,enc_smiles
17,EOS23248,CCOc1ccc(NC(=O)c2ccc(-n3cnnn3)cc2)cc1,0,37,CCOc1ccc(NC(=O)c2ccc(-n3cnnn3)cc2)cc1,"[[2, 2, 4, 13, 28, 13, 13, 13, 24, 3, 2, 24, 1..."
20,EOS86630,COCCn1cc(NC(=O)N2CCN(Cc3cc(C)no3)CC2)cn1,0,40,COCCn1cc(NC(=O)N2CCN(Cc3cc(C)no3)CC2)cn1,"[[2, 4, 2, 2, 14, 28, 13, 13, 24, 3, 2, 24, 17..."
23,EOS317,O=C(c1cccc(N2CCCCS2(=O)=O)c1)N1CCc2ccccc21,0,42,O=C(c1cccc(N2CCCCS2(=O)=O)c1)N1CCc2ccccc21,"[[4, 17, 2, 24, 13, 28, 13, 13, 13, 13, 24, 3,..."
29,EOS49570,Cc1ccccc1[C@@H](NS(=O)(=O)C1CCS(=O)(=O)CC1)C1CC1,0,48,Cc1ccccc1C(NS(=O)(=O)C1CCS(=O)(=O)CC1)C1CC1,"[[2, 13, 28, 13, 13, 13, 13, 13, 28, 2, 24, 3,..."
50,EOS20280,CC(C)C[C@H]1C(=O)N2C[C@@H](N(C)C)C[C@H]2CN1C(=...,0,68,CC(C)CC1C(=O)N2CC(N(C)C)CC2CN1C(=O)CCc1nc(-c2c...,"[[2, 2, 24, 2, 25, 2, 2, 28, 2, 24, 17, 4, 25,..."
...,...,...,...,...,...,...
70537,EOS69640,C#CC(CC)(CC)NC(=O)C1CCN(S(=O)(=O)c2ccc(OC)cc2)CC1,2,49,C#CC(CC)(CC)NC(=O)C1CCN(S(=O)(=O)c2ccc(OC)cc2)CC1,"[[2, 18, 2, 2, 24, 2, 2, 25, 24, 2, 2, 25, 3, ..."
70542,EOS81766,Cc1ccc(CC(=O)N[C@H]2CCc3nc(C)cn3C2)cn1,2,38,Cc1ccc(CC(=O)NC2CCc3nc(C)cn3C2)cn1,"[[2, 13, 28, 13, 13, 13, 24, 2, 2, 24, 17, 4, ..."
70557,EOS39302,CCN(Cc1ccc(Cl)s1)C(=O)CCc1c(C)[nH]c(C)nc1=O,2,43,CCN(Cc1ccc(Cl)s1)C(=O)CCc1c(C)[nH]c(C)nc1=O,"[[2, 2, 3, 24, 2, 13, 28, 13, 13, 13, 24, 8, 2..."
70561,EOS10587,COCCN1CCC(CN(C)S(=O)(=O)c2cccc(C(F)(F)F)c2)C1,2,45,COCCN1CCC(CN(C)S(=O)(=O)c2cccc(C(F)(F)F)c2)C1,"[[2, 4, 2, 2, 3, 28, 2, 2, 2, 24, 2, 3, 24, 2,..."


In [30]:
# tmp[tmp.Id.isin(train_idx)]

,Id,smiles,sol_category,sizes,norm_smiles,enc_smiles


In [16]:
TARGET = ['sol_category']
BATCH_SIZE = int(2048)
ind_id = all_data[all_data.Id.isin(indep_idx)].index
indep_loader = get_loader(np.concatenate(all_data['enc_smiles'].loc[ind_id].to_numpy()).reshape(len(all_data['enc_smiles'].loc[ind_id]),-1),
                          descr_scaled[ind_id],
                          y=all_data[TARGET].loc[ind_id, TARGET].to_numpy(), 
                          with_y=True, 
                          shuffle=False, 
                          batch_size=BATCH_SIZE)

In [17]:
def kappa_val_func(x,y_train_answers,pred_train_probas):
    x1,x2,x3 = x
    local_preds = pred_train_probas**np.array([x1,x2,x3])
    return -round(cohen_kappa_score(y_train_answers, local_preds.argmax(axis=1),weights='quadratic'),4)

In [39]:
from torcheval.metrics import MulticlassAUPRC
from scipy.optimize import minimize
bounds = [(0.01,4)]*3
x0 = [1.0]*3
##V2 - balance the weights
##V3 - lr 1e-4, weight decay
##V4 - lr 1e-4, weight decay, 10 early, smaller neural
##V4A - lr 1e-3, no weight decay, 10 early, smaller neural, no balance, bigger batch 2048, lr 1e-3 
##V5 - same as V6, no rdkit
##V6 - with rdkit
##V6S - use shuffle split instead of kfold
##V7S - with extra features from winning notebooks, except compound id
##V6B - 5 randomizations, no features from winning notebook, new val and train
##V6B_s - 5 randomizations, no features from winning notebook, new val and train, smaller emb 256
##V6B_f - 5 randomizations, no features from winning notebook, new val and train, 512, proper val and ind
TARGET = ['sol_category']
MODEL_VERSION = 'V6B_s2'
EARLY_STOP = 10
BATCH_SIZE = int(2048)
LR = 1e-4
WD = 0.00
EPOCHS = 60
NFOLDS = 3
WH_BALANCE = ''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

indep_loader = get_loader(np.concatenate(all_data['enc_smiles'].loc[ind_id].to_numpy()).reshape(len(all_data['enc_smiles'].loc[ind_id]),-1),
                          descr_scaled[ind_id],
                          y=all_data[TARGET].loc[ind_id, TARGET].to_numpy(), 
                          with_y=True, 
                          shuffle=False, 
                          batch_size=BATCH_SIZE)

for fold in range(5):
    val_ID = []
    with open(f'solubility_valid_id_{fold}.txt', 'r') as fp:
        for line in fp.read().splitlines():
            val_ID.append(line)

    train_ID = []
    with open(f'solubility_train_id_{fold}.txt', 'r') as fp:
        for line in fp.read().splitlines():
            train_ID.append(line)

    train_idx = all_data[all_data.Id.isin(train_ID)].index
    val_idx = all_data[all_data.Id.isin(val_ID)].index
    
    train_loader = get_loader(np.concatenate(all_data['enc_smiles'].loc[train_idx].to_numpy()).reshape(len(all_data['enc_smiles'].loc[train_idx]),-1),
                              descr_scaled[train_idx],
                              y=all_data[TARGET].loc[train_idx, TARGET].to_numpy(), 
                              with_y=True, 
                              shuffle=True, 
                              batch_size=BATCH_SIZE)
    val_loader = get_loader(np.concatenate(all_data['enc_smiles'].loc[val_idx].to_numpy()).reshape(len(all_data['enc_smiles'].loc[val_idx]),-1),
                            descr_scaled[val_idx],
                            y=all_data[TARGET].loc[val_idx, TARGET].to_numpy(), 
                            with_y=True, 
                            shuffle=False, 
                            batch_size=BATCH_SIZE)
    
    curr_name = f"1DCNN_SOL_{fold}_{MODEL_VERSION}_{WH_BALANCE}"
    
    
    neural = CNN1D()

    
    # neural = nn.DataParallel(neural)
    neural = neural.to(device)

    metric = MulticlassAUPRC(num_classes=3)
    metric.to(device)
    # optimizer = AdamW(neural.parameters(), lr=LR, weight_decay=WD)
    optimizer = Adam(neural.parameters(), lr=LR)
    # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=5)
    class_weights=class_weight.compute_class_weight(class_weight='balanced',classes=np.unique(all_data[TARGET].loc[train_idx, TARGET].to_numpy()),y=all_data[TARGET].loc[train_idx, TARGET].to_numpy().flatten())
    class_weights=torch.tensor(class_weights,dtype=torch.float).to(device)
    if WH_BALANCE == 'balanced':
        loss = nn.CrossEntropyLoss(weight=class_weights)
    else:
        loss = nn.CrossEntropyLoss()

    neural_folder = '1dCNN_pytorch_neurals_upd/'

    os.makedirs(neural_folder,exist_ok=True)
    
    res_dict = train_m_v3(neural, optimizer, train_loader, val_loader, loss, EPOCHS, 
                          device, neural_folder+curr_name,metric,early_stop=EARLY_STOP, verbose=True) 
        
    with torch.cuda.device(device):
        torch.cuda.empty_cache()
    
    mpl.rcParams['figure.dpi'] = 150
    print(res_dict['min_loss_epoch'], res_dict['final_val'], res_dict['final_train'], res_dict['val_min'])
    
    with open(os.path.join(neural_folder, curr_name+'_results.txt'), "w") as f:
        f.write(json.dumps(res_dict))

    y1 = []
    y2 = []
    for lg in res_dict['train_loss_history']:
        y1.append(lg)
    for lg in res_dict['val_loss_history']:
        y2.append(lg)
    x = range(1,len(y1)+1)
    plt.plot(x,y1, label = 'train loss')
    plt.plot(x,y2, label = 'val loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(neural_folder+curr_name+'_loss_func.png')
    plt.clf()
    plt.close()
    
    sol_types = ['LOW','MEDIUM','HIGH']
    
    ##find coefs
    neural = CNN1D()
    neural.load_state_dict(torch.load(neural_folder+curr_name,map_location=device))
    neural = neural.to(device)
    neural.eval()
    
    y_answers = []
    pred_probas = []
    with torch.no_grad():
        m = nn.Softmax(dim=1)
        for smile_emb,rdkit_f,y in train_loader:
            out = neural(smile_emb.to(device),rdkit_f.to(device))
            pred = m(out.to(torch.device("cpu"))).detach().numpy()
            y_answers.extend(y.to(torch.device("cpu")).detach().numpy())
            pred_probas.extend(pred)
    y_answers = np.array(y_answers)
    pred_probas = np.array(pred_probas)

    minim_res = minimize(kappa_val_func, x0,args=(y_answers,pred_probas), method='powell',bounds=bounds)
    coefs = np.array(minim_res.x)

    with open(f'kappa_coefs_{curr_name}.txt', 'w') as fp:
        for item in coefs:
            # write each item on a new line
            fp.write("%s\n" % item)
        
    ##val
    y_answers = []
    pred_probas = []
    with torch.no_grad():
        m = nn.Softmax(dim=1)
        for smile_emb,rdkit_f,y in val_loader:
            out = neural(smile_emb.to(device),rdkit_f.to(device))
            pred = m(out.to(torch.device("cpu"))).detach().numpy()
            y_answers.extend(y.to(torch.device("cpu")).detach().numpy())
            pred_probas.extend(pred)
    y_answers = np.array(y_answers)
    pred_probas = np.array(pred_probas)
    
    precs = []
    recs = []
    aps = []
    
    for cl in range(0, 3):
        precision, recall, _ = precision_recall_curve((y_answers==cl).astype(int), pred_probas[:, cl], pos_label=1)
        average_precision = average_precision_score((y_answers==cl).astype(int), pred_probas[:, cl], pos_label=1)
        precs.append(precision)
        recs.append(recall)
        aps.append(average_precision)
    
    print(classification_report(y_answers, pred_probas.argmax(axis=1), target_names=sol_types),
          file=open(neural_folder+'valdiation_classification_report'+curr_name+'.txt', 'a'))

    print(classification_report(y_answers, (pred_probas**coefs).argmax(axis=1), target_names=sol_types),
          file=open(neural_folder+'coef_valdiation_classification_report'+curr_name+'.txt', 'a'))
    
    
    mpl.rcParams['figure.dpi'] = 150
    kappa = round(cohen_kappa_score(y_answers, pred_probas.argmax(axis=1),weights='quadratic'),4)
    adj_kappa = round(cohen_kappa_score(y_answers, (pred_probas**coefs).argmax(axis=1),weights='quadratic'),4)
    plt.title(f'Precision-recall curve, Quadratic kappa {kappa}, adjusted {adj_kappa}', fontsize=14)
    
    for marker in range(3):
        plt.plot(recs[marker], precs[marker], label=f'AUC {sol_types[marker]} = %0.2f' % aps[marker])
    plt.legend(loc='lower right')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.xticks(np.arange(0, 1.1, 0.1), fontsize=9)
    plt.yticks(np.arange(0, 1.1, 0.1), fontsize=9)
    plt.grid(True)
    plt.savefig(neural_folder+f'valdiation_precision_recall_{curr_name}.png', format="png")
    plt.close()

    ##indep
    y_answers = []
    pred_probas = []
    with torch.no_grad():
        m = nn.Softmax(dim=1)
        for smile_emb,rdkit_f,y in indep_loader:
            out = neural(smile_emb.to(device),rdkit_f.to(device))
            pred = m(out.to(torch.device("cpu"))).detach().numpy()
            y_answers.extend(y.to(torch.device("cpu")).detach().numpy())
            pred_probas.extend(pred)
    y_answers = np.array(y_answers)
    pred_probas = np.array(pred_probas)
    precs = []
    recs = []
    aps = []
    
    for cl in range(0, 3):
        precision, recall, _ = precision_recall_curve((y_answers==cl).astype(int), pred_probas[:, cl], pos_label=1)
        average_precision = average_precision_score((y_answers==cl).astype(int), pred_probas[:, cl], pos_label=1)
        precs.append(precision)
        recs.append(recall)
        aps.append(average_precision)
    
    print(classification_report(y_answers, pred_probas.argmax(axis=1), target_names=sol_types),
          file=open(neural_folder+'indep_classification_report'+curr_name+'.txt', 'a'))
    print(classification_report(y_answers, (pred_probas**coefs).argmax(axis=1), target_names=sol_types),
          file=open(neural_folder+'coef_indep_classification_report'+curr_name+'.txt', 'a'))
    
    
    mpl.rcParams['figure.dpi'] = 150
    kappa = round(cohen_kappa_score(y_answers, pred_probas.argmax(axis=1),weights='quadratic'),4)
    adj_kappa = round(cohen_kappa_score(y_answers, (pred_probas**coefs).argmax(axis=1),weights='quadratic'),4)
    plt.title(f'Precision-recall curve, Quadratic kappa {kappa}, adjusted {adj_kappa}', fontsize=14)
    
    for marker in range(3):
        plt.plot(recs[marker], precs[marker], label=f'AUC {sol_types[marker]} = %0.2f' % aps[marker])
    plt.legend(loc='lower right')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.xticks(np.arange(0, 1.1, 0.1), fontsize=9)
    plt.yticks(np.arange(0, 1.1, 0.1), fontsize=9)
    plt.grid(True)
    plt.savefig(neural_folder+f'indep_precision_recall_{curr_name}.png', format="png")
    plt.close()

  0%|                                                                                           | 0/60 [00:00<?, ?it/s]C:\Users\Alexey\miniconda3\envs\work_env\Lib\site-packages\torch\nn\modules\module.py:1511: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
  2%|█▍                                                                                 | 1/60 [00:02<02:48,  2.86s/it]

Epoch [1/60], Loss (train/test): 1.1056157350540161/1.1057534217834473,AP (train/test): 0.3408709466457367/0.3413509130477905


  3%|██▊                                                                                | 2/60 [00:05<02:43,  2.82s/it]

Epoch [2/60], Loss (train/test): 0.9305105805397034/0.9256922006607056,AP (train/test): 0.34381532669067383/0.3445216417312622


  5%|████▏                                                                              | 3/60 [00:08<02:41,  2.83s/it]

...

 50%|█████████████████████████████████████████                                         | 30/60 [01:23<01:22,  2.75s/it]

Epoch [30/60], Loss (train/test): 0.07494799047708511/0.2800823748111725,AP (train/test): 0.9004873037338257/0.33914703130722046


 50%|█████████████████████████████████████████                                         | 30/60 [01:26<01:26,  2.88s/it]

Epoch [31/60], Loss (train/test): 0.06653641909360886/0.2821054458618164,AP (train/test): 0.9342179298400879/0.3412705659866333
Early stopping after epoch 30, model saved at epoch 20, validation loss 0.26137322187423706.
20 0.26137322187423706 0.17965425550937653 0.26137322187423706



C:\Users\Alexey\miniconda3\envs\work_env\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Alexey\miniconda3\envs\work_env\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Alexey\miniconda3\envs\work_env\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Use

In [40]:
# V6bs
import torch.nn.functional as F
from torch.nn import Linear, Conv1d, Embedding, AdaptiveMaxPool1d, AdaptiveAvgPool1d
from torch_geometric.nn import BatchNorm


class CNN1D(torch.nn.Module):
    def __init__(self,
                 conv_filters = 64,
                 conv_layers = 5,
                 emb_dim = 256,
                 kernel_size = 5,
                 linear_neurons=1024):

        super(CNN1D, self).__init__()
        
        self.emb_dim = emb_dim
        
        self.conv_filters = conv_filters

        self.num_conv1d_layers = conv_layers
        
        self.smile_emb = Embedding(num_embeddings=40, embedding_dim=emb_dim)

        self.conv1d_layers = nn.ModuleList()

        for i in range(conv_layers):
            if i == 0:
                self.conv1d_layers.append(
                    Conv1d(in_channels=emb_dim, out_channels=conv_filters * (i+1), kernel_size=kernel_size, stride=1))
            else:
                self.conv1d_layers.append(
                Conv1d(in_channels=conv_filters * i, out_channels=conv_filters * (i+1), kernel_size=kernel_size, stride=1))

        self.maxpool = AdaptiveMaxPool1d(1)
        self.avgpool = AdaptiveAvgPool1d(1)


        self.conv_act = nn.ReLU()
        self.batchnorm = nn.BatchNorm1d(11 + (conv_filters * (conv_layers) * 2))

        # self.actfc = nn.LeakyReLU(negative_slope=0.1)

        # self.gated_mlp = Gated_MLP_block(384)

        self.out_mlp = nn.Sequential(Linear(in_features= 11 + (conv_filters * (conv_layers) * 2), out_features=linear_neurons),
                                     nn.BatchNorm1d(linear_neurons),
                                     nn.LeakyReLU(),
                                     nn.Dropout(0.2),
                                    Linear(in_features=linear_neurons, out_features=linear_neurons //2),
                                     nn.BatchNorm1d(linear_neurons // 2),
                                     nn.LeakyReLU(),
                                     nn.Dropout(0.2),
                                     Linear(in_features=linear_neurons // 2, out_features=linear_neurons //4),
                                     nn.BatchNorm1d(linear_neurons // 4),
                                     nn.LeakyReLU(),
                                     nn.Dropout(0.2),
                                     Linear(in_features=linear_neurons // 4, out_features=linear_neurons // 8),
                                     nn.BatchNorm1d(linear_neurons // 8),
                                     nn.LeakyReLU(),
                                     nn.Dropout(0.2),
                                     )

        self.layer_out = Linear(in_features=linear_neurons // 8, out_features=3)

    def forward(self, smile_vec,rdkit):

        smile_emb = self.smile_emb(smile_vec)

        conv_out = smile_emb.reshape(len(smile_emb), self.emb_dim, -1)

        for i in range(self.num_conv1d_layers):
            conv_out = self.conv1d_layers[i](conv_out)
            conv_out = self.conv_act(conv_out)

        conv_out_max = self.maxpool(conv_out).reshape(len(conv_out), self.conv_filters * self.num_conv1d_layers)
        conv_out_avg = self.avgpool(conv_out).reshape(len(conv_out), self.conv_filters * self.num_conv1d_layers)

        out = torch.cat((conv_out_max,conv_out_avg,rdkit), dim=1)
        out = self.batchnorm(out)

        out = self.out_mlp(out)

        out = self.layer_out(out)

        return out

In [19]:
def kappa_val_func(x,y_train_answers,pred_train_probas):
    x1,x2,x3 = x
    local_preds = pred_train_probas**np.array([x1,x2,x3])
    return -round(cohen_kappa_score(y_train_answers, local_preds.argmax(axis=1),weights='quadratic'),4)

In [20]:
indep_idx = []
with open(f'solubility_indep_ids.txt', 'r') as fp:
    for line in fp.read().splitlines():
        indep_idx.append(line)
ind_id = all_data[all_data.Id.isin(indep_idx)].index

In [23]:
val_ID = []
with open(f'solubility_valid_id_{0}.txt', 'r') as fp:
    for line in fp.read().splitlines():
        val_ID.append(line)

train_ID = []
with open(f'solubility_train_id_{0}.txt', 'r') as fp:
    for line in fp.read().splitlines():
        train_ID.append(line)

uni_id = val_ID+train_ID

train_idx = all_data[all_data.Id.isin(uni_id)].index

train_loader = get_loader(np.concatenate(all_data['enc_smiles'].loc[train_idx].to_numpy()).reshape(len(all_data['enc_smiles'].loc[train_idx]),-1),
                          descr_scaled[train_idx],
                          y=all_data[TARGET].loc[train_idx, TARGET].to_numpy(), 
                          with_y=True, 
                          shuffle=False, 
                          batch_size=BATCH_SIZE)

In [24]:

##V1 - BATCH_SIZE = int(2048) EPOCHS = 60 LR = 1e-4 WD = 0.00
## A version - BATCH_SIZE = int(2048) EPOCHS = 60 LR = 1e-3 WD = 0.00
TARGET = ['sol_category']
MODEL_VERSION = 'V6B_s2'
BATCH_SIZE = int(2048)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WH_BALANCE = ''
all_pred_probas = []
for fold in range(0,5):

    curr_name = f"1DCNN_SOL_{fold}_{MODEL_VERSION}_{WH_BALANCE}"

    neural_folder = '1dCNN_pytorch_neurals_upd/'
    
    neural = CNN1D()
    neural.load_state_dict(torch.load(neural_folder+curr_name,map_location=device))
    neural = neural.to(device)
    neural.eval()

    
    sol_types = ['LOW','MEDIUM','HIGH']
    
    ##find coefs

    y_answers = []
    pred_probas = []
    with torch.no_grad():
        m = nn.Softmax(dim=1)
        for smile_emb,rdkit_f,y in train_loader:
            out = neural(smile_emb.to(device),rdkit_f.to(device))
            pred = m(out.to(torch.device("cpu"))).detach().numpy()
            y_answers.extend(y.to(torch.device("cpu")).detach().numpy())
            pred_probas.extend(pred)
    y_answers = np.array(y_answers)
    all_pred_probas.append(pred_probas)
    


In [25]:
from scipy.optimize import minimize
bounds = [(0.01,4)]*3
x0 = [1.0]*3
pred_probas = np.array(all_pred_probas).mean(axis=0)
minim_res = minimize(kappa_val_func, x0,args=(y_answers,pred_probas), method='powell',bounds=bounds)
coefs = np.array(minim_res.x)

with open(f'kappa_coefs_5_neurals_together_{curr_name}.txt', 'w') as fp:
    for item in coefs:
        # write each item on a new line
        fp.write("%s\n" % item)

In [31]:
all_indep_probas = []
for fold in range(0,5):
    pred_probas_indep = []
    y_answers = []
    curr_name = f"1DCNN_SOL_{fold}_{MODEL_VERSION}_{WH_BALANCE}"

    neural_folder = '1dCNN_pytorch_neurals_upd/'
    
    neural = CNN1D()
    neural.load_state_dict(torch.load(neural_folder+curr_name,map_location=device))
    neural = neural.to(device)
    neural.eval()

    
    sol_types = ['LOW','MEDIUM','HIGH']
    
    ##find coefs

    y_answers = []
    pred_probas = []
    with torch.no_grad():
        m = nn.Softmax(dim=1)
        for smile_emb,rdkit_f,y in indep_loader:
            out = neural(smile_emb.to(device),rdkit_f.to(device))
            pred = m(out.to(torch.device("cpu"))).detach().numpy()
            y_answers.extend(y.to(torch.device("cpu")).detach().numpy())
            pred_probas_indep.extend(pred)
            
    all_indep_probas.append(np.array(pred_probas_indep))

y_answers = np.array(y_answers)
pred_probas = np.array(all_indep_probas).mean(axis=0)

In [34]:
##indep
precs = []
recs = []
aps = []

for cl in range(0, 3):
    precision, recall, _ = precision_recall_curve((y_answers==cl).astype(int), pred_probas[:, cl], pos_label=1)
    average_precision = average_precision_score((y_answers==cl).astype(int), pred_probas[:, cl], pos_label=1)
    precs.append(precision)
    recs.append(recall)
    aps.append(average_precision)

print(classification_report(y_answers, pred_probas.argmax(axis=1), target_names=sol_types),
      file=open(neural_folder+'5_split_indep_classification_report'+curr_name+'.txt', 'a'))
print(classification_report(y_answers, (pred_probas**coefs).argmax(axis=1), target_names=sol_types),
      file=open(neural_folder+'5_split_coef_indep_classification_report'+curr_name+'.txt', 'a'))


mpl.rcParams['figure.dpi'] = 150
kappa = round(cohen_kappa_score(y_answers, pred_probas.argmax(axis=1),weights='quadratic'),4)
adj_kappa = round(cohen_kappa_score(y_answers, (pred_probas**coefs).argmax(axis=1),weights='quadratic'),4)
plt.title(f'Precision-recall curve, Quadratic kappa {kappa}, adjusted {adj_kappa}', fontsize=14)

for marker in range(3):
    plt.plot(recs[marker], precs[marker], label=f'AUC {sol_types[marker]} = %0.2f' % aps[marker])
plt.legend(loc='lower right')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.xticks(np.arange(0, 1.1, 0.1), fontsize=9)
plt.yticks(np.arange(0, 1.1, 0.1), fontsize=9)
plt.grid(True)
plt.savefig(neural_folder+f'indep_precision_recall_5_split_{curr_name}.png', format="png")
plt.close()

C:\Users\Alexey\miniconda3\envs\work_env\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Alexey\miniconda3\envs\work_env\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Alexey\miniconda3\envs\work_env\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [39]:
test_data = pd.read_csv('test.csv')

In [44]:

# all_data['smiles'] = all_data['smiles'].apply(normalize_smile)
# all_data['enc_smiles'] = all_data['smiles'].apply(encode_smile)
test_descr = []
# calc some descriptors
for m in tqdm(test_data.smiles):
    test_descr.append([Descriptors.MolLogP(Chem.MolFromSmiles(m)),
                  Descriptors.TPSA(Chem.MolFromSmiles(m)),
                  Descriptors.NHOHCount(Chem.MolFromSmiles(m)),
                  Descriptors.NOCount(Chem.MolFromSmiles(m)),
                  Descriptors.NumHAcceptors(Chem.MolFromSmiles(m)),
                  Descriptors.NumHDonors(Chem.MolFromSmiles(m)),
                  Descriptors.NumRotatableBonds(Chem.MolFromSmiles(m)),
                  Descriptors.NumHeteroatoms(Chem.MolFromSmiles(m)),
                  Descriptors.FractionCSP3(Chem.MolFromSmiles(m)),
                  Descriptors.ExactMolWt(Chem.MolFromSmiles(m)),
                  rdMolDescriptors.CalcNumAromaticRings(Chem.MolFromSmiles(m))])

test_descr = np.asarray(test_descr)
scaler_descr = torch.load('rdkit_descriptor_scaler_sol')
test_descr_scaled = scaler_descr.transform(test_descr)

100%|███████████████████████████████████████████████████████████████████████████| 30307/30307 [01:13<00:00, 414.07it/s]


In [45]:
test_data['norm_smiles'] = test_data['smiles'].apply(normalize_smile)
test_data['enc_smiles'] = test_data['norm_smiles'].apply(encode_smile)

In [46]:
test_loader = get_loader(np.concatenate(test_data['enc_smiles'].to_numpy()).reshape(len(test_data['enc_smiles']),-1),
                          test_descr_scaled,
                          with_y=False, 
                          shuffle=False, 
                          batch_size=BATCH_SIZE)

In [50]:
coefs = []

with open(f'kappa_coefs_5_neurals_together_{curr_name}.txt', 'r') as fp:
    for line in fp.read().splitlines():
        coefs.append(float(line))

coefs = np.array(coefs)

In [48]:

##V1 - BATCH_SIZE = int(2048) EPOCHS = 60 LR = 1e-4 WD = 0.00
## A version - BATCH_SIZE = int(2048) EPOCHS = 60 LR = 1e-3 WD = 0.00
TARGET = ['sol_category']
MODEL_VERSION = 'V6B_s2'
BATCH_SIZE = int(2048)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WH_BALANCE = ''
all_pred_probas = []
for fold in range(0,5):

    curr_name = f"1DCNN_SOL_{fold}_{MODEL_VERSION}_{WH_BALANCE}"

    neural_folder = '1dCNN_pytorch_neurals_upd/'
    
    neural = CNN1D()
    neural.load_state_dict(torch.load(neural_folder+curr_name,map_location=device))
    neural = neural.to(device)
    neural.eval()

    
    sol_types = ['LOW','MEDIUM','HIGH']
    
    ##find coefs

    y_answers = []
    pred_probas = []
    with torch.no_grad():
        m = nn.Softmax(dim=1)
        for smile_emb,rdkit_f in test_loader:
            out = neural(smile_emb.to(device),rdkit_f.to(device))
            pred = m(out.to(torch.device("cpu"))).detach().numpy()
            pred_probas.extend(pred)
    all_pred_probas.append(pred_probas)
    


In [53]:
test_pred_probas = np.array(all_pred_probas).mean(axis=0)
answers = (test_pred_probas**coefs).argmax(axis=1)

In [54]:
template = pd.read_csv('submission_template_rdm.csv')
template['Id'] = test_data['Id']
template['pred'] = answers
template.set_index('Id',inplace=True,drop=True)
template.to_csv('kaggle_1dCNN_5_split_coef.csv')